# Neighborhood Reduction (NR) Ablation: Paired Comparison

This notebook compares two ALNS experiments on the same **20 MD-DARP instances x 20 seeds**:

- **Base**: the original ALNS run (`./run_base_20/`)
- **NR**: ALNS run with the `--NR` (neighborhood reduction) flag activated (`./run_nr/`)

Because both experiments use the *same* instance/seed grid, every run in one experiment has a
direct counterpart in the other (`instance`, `seed`) pair, so the analysis is built around
**paired comparisons** (same instance, same seed, Base vs. NR) rather than independent samples.

The analysis covers:

1. Loading both experiments with a shared, robust loader.
2. Run-level and instance-level summaries for each variant.
3. Direct pairing of runs and RPD computed against a **joint** best-known reference (across both variants).
4. Instance-level and global paired comparison tables.
5. Paired significance testing (Wilcoxon signed-rank) on objective and runtime.
6. Win/loss/tie breakdown.
7. Visual comparisons: objective, RPD, runtime, convergence curves, time-to-best.
8. Destroy/repair operator selection: Base vs. NR.
9. CSV exports of the main comparison tables.

In [1]:
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from IPython.display import display, Markdown

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

BASE_DIR = Path("./run_base_20")
NR_DIR = Path("./run_nr")

VARIANT_DIRS = {
    "Base": BASE_DIR,
    "NR": NR_DIR,
}

EXPECTED_INSTANCES = [
    "md1a", "md2a", "md3a", "md4a", "md5a",
    "md6a", "md7a", "md8a", "md9a", "md10a",
    "md1b", "md2b", "md3b", "md4b", "md5b",
    "md6b", "md7b", "md8b", "md9b", "md10b",
]

EXPECTED_SEEDS = range(1, 21)

DESTROY_METHODS = {
    0: "Random",
    1: "Worst",
    2: "Shaw",
}

REPAIR_METHODS = {
    0: "Greedy",
    1: "Regret-2",
    2: "Regret-3",
}

COMPARISON_DIR = Path("../results/nr")
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

# Plot configuration
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 15
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["lines.linewidth"] = 2
plt.rcParams["figure.dpi"] = 120

VARIANT_COLORS = {"Base": "tab:blue", "NR": "tab:orange"}

for name, d in VARIANT_DIRS.items():
    print(f"{name:>5} log dir:    {(d / 'alns_logs').resolve()}")
    print(f"{name:>5} output dir: {(d / 'outputs').resolve()}")


 Base log dir:    /home/guillem/MD-DARP-Research/experiments/notebooks/run_base_20/alns_logs
 Base output dir: /home/guillem/MD-DARP-Research/experiments/notebooks/run_base_20/outputs
   NR log dir:    /home/guillem/MD-DARP-Research/experiments/notebooks/run_nr/alns_logs
   NR output dir: /home/guillem/MD-DARP-Research/experiments/notebooks/run_nr/outputs


## 1. Reusable loaders

The same parsing/loading logic is used for both experiment directories, so results are
guaranteed to be processed identically. Filenames are parsed with **optional** underscores
around the instance name (see note above) so both naming conventions are supported.

In [2]:
# ---------------------------------------------------------
# Flexible filename patterns (optional underscores around instance name)
# ---------------------------------------------------------

CONV_PATTERN = re.compile(r"^log_?(.+?)_?seed(\d+)_convergence\.csv$")
WEIGHT_PATTERN = re.compile(r"^log_?(.+?)_?seed(\d+)_weights_evolution\.csv$")
SOLUTION_PATTERN = re.compile(r"^sol_?(.+?)_?seed(\d+)\.json$")


def parse_name(pattern, path):
    match = pattern.match(path.name)
    if not match:
        return None, None
    return match.group(1), int(match.group(2))


def discover_files(directory):
    log_dir = directory / "alns_logs"
    out_dir = directory / "outputs"
    conv_files = sorted(log_dir.glob("*convergence.csv"))
    weight_files = sorted(log_dir.glob("*weights_evolution.csv"))
    solution_files = sorted(out_dir.glob("*.json"))
    return conv_files, weight_files, solution_files


def check_completeness(name, discovered_runs):
    missing = [
        (inst, seed)
        for inst in EXPECTED_INSTANCES
        for seed in EXPECTED_SEEDS
        if (inst, seed) not in discovered_runs
    ]
    unexpected = [
        run for run in discovered_runs
        if run[0] not in EXPECTED_INSTANCES or run[1] not in EXPECTED_SEEDS
    ]

    print(f"[{name}] unique runs discovered: {len(discovered_runs)}")
    if missing:
        print(f"[{name}] missing expected runs ({len(missing)}):")
        print(missing[:30])
        if len(missing) > 30:
            print(f"... and {len(missing) - 30} more")
    if unexpected:
        print(f"[{name}] unexpected runs ({len(unexpected)}):")
        print(unexpected[:30])
    if not missing and not unexpected:
        print(f"[{name}] all expected {len(EXPECTED_INSTANCES)} x {len(list(EXPECTED_SEEDS))} runs found.")
    print()


def load_convergence(conv_files, variant):
    required_cols = {
        "iteration", "elapsed_time", "current_obj", "best_obj",
        "best_feasible_obj", "destroy_op", "repair_op", "reward",
        "temperature", "segment",
    }
    numeric_cols = list(required_cols)

    frames = []
    for path in conv_files:
        instance, seed = parse_name(CONV_PATTERN, path)
        if instance is None:
            continue
        try:
            df = pd.read_csv(path)
        except Exception as exc:
            warnings.warn(f"Could not read {path}: {exc}")
            continue

        missing_cols = required_cols - set(df.columns)
        if missing_cols:
            warnings.warn(f"{path.name}: missing columns {sorted(missing_cols)}")
            continue

        df["instance"] = instance
        df["seed"] = seed
        df["run_id"] = f"{instance}_seed{seed}"
        df["variant"] = variant

        for col in numeric_cols:
            df[col] = pd.to_numeric(df[col], errors="coerce")

        df["best_feasible_obj"] = df["best_feasible_obj"].replace([np.inf, -np.inf], np.nan)
        frames.append(df)

    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def load_weights(weight_files, variant):
    required_cols = {
        "segment", "elapsed_time",
        "destroy_weight_0", "destroy_weight_1", "destroy_weight_2",
        "repair_weight_0", "repair_weight_1", "repair_weight_2",
    }

    frames = []
    for path in weight_files:
        instance, seed = parse_name(WEIGHT_PATTERN, path)
        if instance is None:
            continue
        try:
            df = pd.read_csv(path)
        except Exception as exc:
            warnings.warn(f"Could not read {path}: {exc}")
            continue

        missing_cols = required_cols - set(df.columns)
        if missing_cols:
            warnings.warn(f"{path.name}: missing columns {sorted(missing_cols)}")
            continue

        df["instance"] = instance
        df["seed"] = seed
        df["run_id"] = f"{instance}_seed{seed}"
        df["variant"] = variant

        for col in required_cols:
            df[col] = pd.to_numeric(df[col], errors="coerce")

        frames.append(df)

    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def load_solutions(solution_files, variant):
    rows = []
    for path in solution_files:
        instance, seed = parse_name(SOLUTION_PATTERN, path)
        if instance is None:
            continue
        try:
            with open(path, "r") as f:
                data = json.load(f)
        except Exception as exc:
            warnings.warn(f"Could not read {path}: {exc}")
            continue

        summary = data.get("summary", {}) if isinstance(data, dict) else {}

        rows.append({
            "instance": instance,
            "seed": seed,
            "run_id": f"{instance}_seed{seed}",
            "variant": variant,
            "json_objective": summary.get("objective"),
            "json_time_sec": summary.get("time_sec"),
            "status": summary.get("status"),
            "mip_gap": summary.get("mip_gap"),
            "num_requests": summary.get("num_requests"),
            "num_vehicles": summary.get("num_vehicles"),
        })

    df = pd.DataFrame(rows)
    if not df.empty:
        for col in ["json_objective", "json_time_sec", "mip_gap", "num_requests", "num_vehicles"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def build_run_summary(df_conv_all, df_solutions, variant):
    if df_conv_all.empty:
        return pd.DataFrame()

    rows = []
    for run_id, g in df_conv_all.groupby("run_id", sort=False):
        g = g.sort_values("iteration")
        feasible = g["best_feasible_obj"].dropna()

        rows.append({
            "run_id": run_id,
            "instance": g["instance"].iloc[0],
            "seed": int(g["seed"].iloc[0]),
            "variant": variant,
            "iterations": len(g),
            "elapsed_time_sec": g["elapsed_time"].max(),
            "initial_objective": g["current_obj"].iloc[0],
            "final_current_objective": g["current_obj"].iloc[-1],
            "best_objective": g["best_obj"].min(),
            "best_feasible_objective": feasible.min() if not feasible.empty else np.nan,
            "best_iteration": g.loc[g["best_obj"].idxmin(), "iteration"],
            "best_time_sec": g.loc[g["best_obj"].idxmin(), "elapsed_time"],
            "final_temperature": g["temperature"].iloc[-1],
            "segments": g["segment"].max() + 1 if g["segment"].notna().any() else np.nan,
        })

    df_runs = pd.DataFrame(rows)

    if not df_solutions.empty:
        df_runs = df_runs.merge(
            df_solutions.drop(columns=["variant"]),
            on=["run_id", "instance", "seed"],
            how="left",
        )

    return df_runs


## 2. Discover and load both experiments

In [3]:
# ---------------------------------------------------------
# File discovery and completeness check
# ---------------------------------------------------------

files_by_variant = {}
for name, d in VARIANT_DIRS.items():
    conv_files, weight_files, solution_files = discover_files(d)
    files_by_variant[name] = (conv_files, weight_files, solution_files)

    print(f"[{name}] convergence logs found: {len(conv_files)}")
    print(f"[{name}] weight logs found:      {len(weight_files)}")
    print(f"[{name}] solution JSONs found:   {len(solution_files)}")

    discovered_runs = sorted({
        parse_name(CONV_PATTERN, p) for p in conv_files
    } - {(None, None)})
    check_completeness(name, discovered_runs)


[Base] convergence logs found: 0
[Base] weight logs found:      0
[Base] solution JSONs found:   0
[Base] unique runs discovered: 0
[Base] missing expected runs (400):
[('md1a', 1), ('md1a', 2), ('md1a', 3), ('md1a', 4), ('md1a', 5), ('md1a', 6), ('md1a', 7), ('md1a', 8), ('md1a', 9), ('md1a', 10), ('md1a', 11), ('md1a', 12), ('md1a', 13), ('md1a', 14), ('md1a', 15), ('md1a', 16), ('md1a', 17), ('md1a', 18), ('md1a', 19), ('md1a', 20), ('md2a', 1), ('md2a', 2), ('md2a', 3), ('md2a', 4), ('md2a', 5), ('md2a', 6), ('md2a', 7), ('md2a', 8), ('md2a', 9), ('md2a', 10)]
... and 370 more

[NR] convergence logs found: 0
[NR] weight logs found:      0
[NR] solution JSONs found:   0
[NR] unique runs discovered: 0
[NR] missing expected runs (400):
[('md1a', 1), ('md1a', 2), ('md1a', 3), ('md1a', 4), ('md1a', 5), ('md1a', 6), ('md1a', 7), ('md1a', 8), ('md1a', 9), ('md1a', 10), ('md1a', 11), ('md1a', 12), ('md1a', 13), ('md1a', 14), ('md1a', 15), ('md1a', 16), ('md1a', 17), ('md1a', 18), ('md1a', 

In [4]:
# ---------------------------------------------------------
# Load convergence / weights / solutions for both variants
# ---------------------------------------------------------

df_conv = {}
df_weights = {}
df_solutions = {}

for name in VARIANT_DIRS:
    conv_files, weight_files, solution_files = files_by_variant[name]
    df_conv[name] = load_convergence(conv_files, name)
    df_weights[name] = load_weights(weight_files, name)
    df_solutions[name] = load_solutions(solution_files, name)

    n_runs = df_conv[name]["run_id"].nunique() if not df_conv[name].empty else 0
    print(f"[{name}] loaded {n_runs} runs, {len(df_conv[name]):,} convergence rows, "
          f"{len(df_weights[name]):,} weight rows, {len(df_solutions[name])} solution JSONs.")

df_conv_base, df_conv_nr = df_conv["Base"], df_conv["NR"]
df_weights_base, df_weights_nr = df_weights["Base"], df_weights["NR"]
df_solutions_base, df_solutions_nr = df_solutions["Base"], df_solutions["NR"]

df_conv_all = pd.concat([df_conv_base, df_conv_nr], ignore_index=True)
display(df_conv_all.head())


[Base] loaded 0 runs, 0 convergence rows, 0 weight rows, 0 solution JSONs.
[NR] loaded 0 runs, 0 convergence rows, 0 weight rows, 0 solution JSONs.


""


## 3. Run-level summaries for each variant

In [5]:
# ---------------------------------------------------------
# Build one summary row per run, per variant, then combine
# ---------------------------------------------------------

df_runs_base = build_run_summary(df_conv_base, df_solutions_base, "Base")
df_runs_nr = build_run_summary(df_conv_nr, df_solutions_nr, "NR")

df_runs_all = pd.concat([df_runs_base, df_runs_nr], ignore_index=True)

df_runs_all["regime"] = df_runs_all["instance"].str[-1].str.upper()
df_runs_all["regime_order"] = df_runs_all["regime"].map({"A": 0, "B": 1})
df_runs_all["size_id"] = (
    df_runs_all["instance"].str.extract(r"md(\d+)[ab]", expand=False).astype(int)
)
df_runs_all = df_runs_all.sort_values(
    ["regime_order", "size_id"]
).drop(columns="regime_order")

print(f"Base runs: {len(df_runs_base)}  |  NR runs: {len(df_runs_nr)}")
display(df_runs_all.head())


KeyError: 'instance'

## 4. Pair runs by `(instance, seed)`

Since both experiments use the same instance/seed grid, every run has a direct counterpart.
The merge below is an **inner join** on `(instance, seed)`, so it only keeps runs that exist
in both experiments — any asymmetric missing runs are reported first.

In [ ]:
# ---------------------------------------------------------
# Pair runs directly by (instance, seed)
# ---------------------------------------------------------

base_keys = set(zip(df_runs_base["instance"], df_runs_base["seed"]))
nr_keys = set(zip(df_runs_nr["instance"], df_runs_nr["seed"]))

only_base = base_keys - nr_keys
only_nr = nr_keys - base_keys

print(f"Runs only in Base: {len(only_base)}")
if only_base:
    print(sorted(only_base)[:20])
print(f"Runs only in NR:   {len(only_nr)}")
if only_nr:
    print(sorted(only_nr)[:20])

df_paired = df_runs_base.merge(
    df_runs_nr,
    on=["instance", "seed"],
    how="inner",
    suffixes=("_base", "_nr"),
)

df_paired["regime"] = df_paired["instance"].str[-1].str.upper()
df_paired["size_id"] = (
    df_paired["instance"].str.extract(r"md(\d+)[ab]", expand=False).astype(int)
)

print(f"\nPaired runs: {len(df_paired)} (out of {len(df_runs_base)} Base / {len(df_runs_nr)} NR)")
display(df_paired.head())


## 5. Joint RPD reference

For a fair comparison, RPD is computed relative to the **best feasible objective found by
either variant** on that instance (i.e. the joint best-known), rather than a per-variant
reference. This way, a variant that is simply "worse everywhere" does not get an artificially
favourable RPD from being compared only against itself.

In [ ]:
# ---------------------------------------------------------
# Joint best-known reference per instance (across both variants)
# ---------------------------------------------------------

joint_reference = (
    df_runs_all.groupby("instance")["best_feasible_objective"]
    .min()
    .rename("instance_best")
    .reset_index()
)

df_runs_all = df_runs_all.merge(joint_reference, on="instance", how="left")
df_runs_all["RPD_percent"] = (
    100 * (df_runs_all["best_feasible_objective"] - df_runs_all["instance_best"])
    / df_runs_all["instance_best"]
)

df_paired = df_paired.merge(joint_reference, on="instance", how="left")
df_paired["RPD_percent_base"] = (
    100 * (df_paired["best_feasible_objective_base"] - df_paired["instance_best"])
    / df_paired["instance_best"]
)
df_paired["RPD_percent_nr"] = (
    100 * (df_paired["best_feasible_objective_nr"] - df_paired["instance_best"])
    / df_paired["instance_best"]
)

# Paired deltas (NR - Base): negative delta_objective means NR found a BETTER (lower) objective
df_paired["delta_objective"] = (
    df_paired["best_feasible_objective_nr"] - df_paired["best_feasible_objective_base"]
)
df_paired["delta_pct_objective"] = (
    100 * df_paired["delta_objective"] / df_paired["best_feasible_objective_base"]
)
df_paired["delta_RPD"] = df_paired["RPD_percent_nr"] - df_paired["RPD_percent_base"]

df_paired["delta_time_sec"] = df_paired["elapsed_time_sec_nr"] - df_paired["elapsed_time_sec_base"]
df_paired["delta_pct_time"] = 100 * df_paired["delta_time_sec"] / df_paired["elapsed_time_sec_base"]

display(
    df_paired[
        [
            "instance", "seed",
            "best_feasible_objective_base", "best_feasible_objective_nr", "delta_objective", "delta_pct_objective",
            "elapsed_time_sec_base", "elapsed_time_sec_nr", "delta_pct_time",
        ]
    ].head(10).round(4)
)


## 6. Instance-level paired comparison table

In [ ]:
# ---------------------------------------------------------
# Instance-level comparison: Base vs. NR side by side
# ---------------------------------------------------------

instance_comparison = (
    df_paired
    .groupby(["instance", "regime", "size_id"])
    .agg(
        runs=("seed", "count"),
        base_mean_obj=("best_feasible_objective_base", "mean"),
        nr_mean_obj=("best_feasible_objective_nr", "mean"),
        base_mean_RPD=("RPD_percent_base", "mean"),
        nr_mean_RPD=("RPD_percent_nr", "mean"),
        base_mean_runtime=("elapsed_time_sec_base", "mean"),
        nr_mean_runtime=("elapsed_time_sec_nr", "mean"),
        mean_delta_pct_objective=("delta_pct_objective", "mean"),
        mean_delta_pct_time=("delta_pct_time", "mean"),
    )
    .reset_index()
    .sort_values(["regime", "size_id"])
)

instance_comparison["objective_improvement_pct"] = -instance_comparison["mean_delta_pct_objective"]

display(instance_comparison.round(4))


## 7. Global aggregate comparison

In [ ]:
# ---------------------------------------------------------
# Global aggregate statistics: Base vs. NR
# ---------------------------------------------------------

global_rows = []
for name, df in [("Base", df_runs_base), ("NR", df_runs_nr)]:
    merged = df.merge(joint_reference, on="instance", how="left")
    merged["RPD_percent"] = (
        100 * (merged["best_feasible_objective"] - merged["instance_best"])
        / merged["instance_best"]
    )
    global_rows.append({
        "variant": name,
        "runs": len(merged),
        "instances": merged["instance"].nunique(),
        "mean_RPD_%": merged["RPD_percent"].mean(),
        "median_RPD_%": merged["RPD_percent"].median(),
        "mean_runtime_sec": merged["elapsed_time_sec"].mean(),
        "median_runtime_sec": merged["elapsed_time_sec"].median(),
        "mean_objective": merged["best_feasible_objective"].mean(),
        "feasible_runs_%": 100 * merged["best_feasible_objective"].notna().mean(),
    })

global_comparison = pd.DataFrame(global_rows)
display(global_comparison.round(4))


## 8. Paired statistical significance tests

The Wilcoxon signed-rank test is used since it is the natural paired, non-parametric test for
matched (instance, seed) observations. It is run per instance (20 paired seeds) and globally
(all 400 paired runs, treated as an exploratory pooled test — note that pooling across
instances of very different scale/difficulty is not strictly i.i.d., so per-instance results
should be treated as primary).

In [ ]:
# ---------------------------------------------------------
# Wilcoxon signed-rank tests: objective and runtime, per instance + global
# ---------------------------------------------------------

def paired_wilcoxon(base_vals, nr_vals):
    base_vals = np.asarray(base_vals, dtype=float)
    nr_vals = np.asarray(nr_vals, dtype=float)
    mask = ~(np.isnan(base_vals) | np.isnan(nr_vals))
    base_vals, nr_vals = base_vals[mask], nr_vals[mask]

    if len(base_vals) < 2 or np.allclose(base_vals, nr_vals):
        return np.nan, np.nan, mask.sum()

    try:
        stat, p = stats.wilcoxon(base_vals, nr_vals)
    except ValueError:
        stat, p = np.nan, np.nan
    return stat, p, mask.sum()


test_rows = []
for instance, g in df_paired.groupby("instance"):
    stat_obj, p_obj, n_obj = paired_wilcoxon(
        g["best_feasible_objective_base"], g["best_feasible_objective_nr"]
    )
    stat_time, p_time, n_time = paired_wilcoxon(
        g["elapsed_time_sec_base"], g["elapsed_time_sec_nr"]
    )
    test_rows.append({
        "instance": instance,
        "n_pairs": n_obj,
        "objective_wilcoxon_stat": stat_obj,
        "objective_p_value": p_obj,
        "runtime_wilcoxon_stat": stat_time,
        "runtime_p_value": p_time,
    })

significance_table = pd.DataFrame(test_rows)
significance_table["objective_significant_5pct"] = significance_table["objective_p_value"] < 0.05
significance_table["runtime_significant_5pct"] = significance_table["runtime_p_value"] < 0.05

display(significance_table.round(5))

stat_obj_g, p_obj_g, n_obj_g = paired_wilcoxon(
    df_paired["best_feasible_objective_base"], df_paired["best_feasible_objective_nr"]
)
stat_time_g, p_time_g, n_time_g = paired_wilcoxon(
    df_paired["elapsed_time_sec_base"], df_paired["elapsed_time_sec_nr"]
)

print(f"\nGlobal pooled test (n={n_obj_g} pairs) — objective: stat={stat_obj_g:.2f}, p={p_obj_g:.3g}")
print(f"Global pooled test (n={n_time_g} pairs) — runtime:   stat={stat_time_g:.2f}, p={p_time_g:.3g}")


## 9. Win / loss / tie breakdown

For each paired (instance, seed) run, "better" means a lower (best feasible) objective, and
separately, a shorter runtime.

In [ ]:
# ---------------------------------------------------------
# Win/loss/tie counts: NR vs Base
# ---------------------------------------------------------

TOL = 1e-6

def classify(delta, tol=TOL):
    if pd.isna(delta):
        return "N/A"
    if delta < -tol:
        return "NR better"
    if delta > tol:
        return "Base better"
    return "Tie"

df_paired["objective_outcome"] = df_paired["delta_objective"].apply(classify)
df_paired["runtime_outcome"] = df_paired["delta_time_sec"].apply(classify)

print("Objective outcome (all 400 paired runs):")
display(df_paired["objective_outcome"].value_counts().rename("count").to_frame())

print("\nRuntime outcome (all 400 paired runs):")
display(df_paired["runtime_outcome"].value_counts().rename("count").to_frame())

win_loss_by_instance = (
    df_paired
    .groupby("instance")["objective_outcome"]
    .value_counts()
    .unstack(fill_value=0)
    .reindex(columns=["NR better", "Base better", "Tie", "N/A"], fill_value=0)
)

print("\nObjective win/loss by instance:")
display(win_loss_by_instance)


## 10. Visualization — paired objective comparison (per instance)

In [ ]:
# ---------------------------------------------------------
# Slope chart: mean best-feasible objective, Base vs. NR, per instance
# ---------------------------------------------------------

plot_df = instance_comparison.sort_values(["size_id", "regime"]).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 8))

y = np.arange(len(plot_df))

for i, row in plot_df.iterrows():
    ax.plot(
        [row["base_mean_obj"], row["nr_mean_obj"]],
        [i, i],
        color="grey",
        alpha=0.6,
        zorder=1,
    )

ax.scatter(plot_df["base_mean_obj"], y, color=VARIANT_COLORS["Base"], label="Base", zorder=2)
ax.scatter(plot_df["nr_mean_obj"], y, color=VARIANT_COLORS["NR"], label="NR", zorder=2)

ax.set_yticks(y)
ax.set_yticklabels(plot_df["instance"])
ax.set_xlabel("Mean best feasible objective")
ax.set_ylabel("Instance")
ax.set_title("Mean Objective: Base vs. NR (per instance)")
ax.legend()

plt.tight_layout()
plt.show()


## 11. Visualization — RPD distribution comparison

In [ ]:
# ---------------------------------------------------------
# RPD distribution: Base vs. NR (overall and by regime)
# ---------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].boxplot(
    [df_paired["RPD_percent_base"].dropna(), df_paired["RPD_percent_nr"].dropna()],
    labels=["Base", "NR"],
)
axes[0].set_title("RPD Distribution (all runs)")
axes[0].set_ylabel("RPD vs. joint best (%)")

regime_data = []
regime_labels = []
for regime in ["A", "B"]:
    for variant, col in [("Base", "RPD_percent_base"), ("NR", "RPD_percent_nr")]:
        vals = df_paired.loc[df_paired["regime"] == regime, col].dropna()
        regime_data.append(vals)
        regime_labels.append(f"{regime}\n{variant}")

bp = axes[1].boxplot(regime_data, labels=regime_labels, patch_artist=True)
colors = [VARIANT_COLORS["Base"], VARIANT_COLORS["NR"]] * 2
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.5)

axes[1].set_title("RPD by Time-Window Regime")
axes[1].set_ylabel("RPD vs. joint best (%)")

plt.tight_layout()
plt.show()


## 12. Visualization — runtime comparison

In [ ]:
# ---------------------------------------------------------
# Runtime comparison: Base vs. NR
# ---------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].boxplot(
    [df_paired["elapsed_time_sec_base"].dropna(), df_paired["elapsed_time_sec_nr"].dropna()],
    labels=["Base", "NR"],
)
axes[0].set_title("Runtime Distribution (all runs)")
axes[0].set_ylabel("Runtime (s)")

plot_df = instance_comparison.sort_values("size_id")
axes[1].plot(plot_df["size_id"], plot_df["base_mean_runtime"], marker="o",
             color=VARIANT_COLORS["Base"], label="Base")
axes[1].plot(plot_df["size_id"], plot_df["nr_mean_runtime"], marker="o",
             color=VARIANT_COLORS["NR"], label="NR")
axes[1].set_title("Mean Runtime by Instance Size")
axes[1].set_xlabel("Instance size index")
axes[1].set_ylabel("Mean runtime (s)")
axes[1].legend()

plt.tight_layout()
plt.show()


## 13. Aggregate convergence curve comparison

Convergence is normalized by search progress (0-100% of each run's iterations) and RPD is
computed against the same joint best-known reference used above, so the two curves are
directly comparable.

In [ ]:
# ---------------------------------------------------------
# Normalized, aggregated convergence curves: Base vs. NR
# ---------------------------------------------------------

conv_for_curve = df_conv_all.merge(joint_reference, on="instance", how="left")

conv_for_curve["feasible_RPD"] = (
    100 * (conv_for_curve["best_feasible_obj"] - conv_for_curve["instance_best"])
    / conv_for_curve["instance_best"]
)

conv_for_curve["iteration_progress"] = (
    conv_for_curve.groupby("run_id")["iteration"]
    .transform(lambda x: 100 * x / max(x.max(), 1))
)

conv_for_curve["progress_bin"] = np.floor(conv_for_curve["iteration_progress"] / 2) * 2

aggregate_curve = (
    conv_for_curve
    .dropna(subset=["feasible_RPD"])
    .groupby(["variant", "progress_bin"])
    .agg(
        median_RPD=("feasible_RPD", "median"),
        q25_RPD=("feasible_RPD", lambda x: x.quantile(0.25)),
        q75_RPD=("feasible_RPD", lambda x: x.quantile(0.75)),
    )
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 6))

for variant in ["Base", "NR"]:
    g = aggregate_curve[aggregate_curve["variant"] == variant]
    ax.plot(g["progress_bin"], g["median_RPD"], label=f"{variant} (median)",
            color=VARIANT_COLORS[variant])
    ax.fill_between(g["progress_bin"], g["q25_RPD"], g["q75_RPD"],
                     alpha=0.15, color=VARIANT_COLORS[variant])

ax.set_xlabel("Normalized search progress (%)")
ax.set_ylabel("Best feasible RPD vs. joint best (%)")
ax.set_title("Aggregate ALNS Convergence: Base vs. NR")
ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------------------------------------
# Table: Best Objective Across ALL Instances at Time Cutoffs
#                 Base columns | NR columns
# ---------------------------------------------------------

def build_all_instances_time_table(df_conv, time_points=[0.1, 1, 2, 5, 10, 60]):
    """
    Generates an instance-level summary table across all seeds
    at specific time cutoffs.

    Each instance occupies one row:
        Instance | Base_0.1s | Base_1s | ... | NR_0.1s | NR_1s | ...

    The value is the mean best objective across seeds.
    If no seed has found a feasible solution at a cutoff, '*' is added.
    """

    rows = []

    df_sorted = df_conv.sort_values(
        ["instance", "variant", "seed", "elapsed_time"]
    )

    # One row per instance
    for instance, instance_group in df_sorted.groupby("instance", sort=False):

        row_data = {
            "Instance": instance
        }

        # First Base, then NR
        for variant in ["Base", "NR"]:

            variant_group = instance_group[
                instance_group["variant"].str.upper() == variant.upper()
            ]

            seed_groups = variant_group.groupby("seed")

            for t in time_points:

                vals_at_t = []
                feas_flags = []

                for seed, s_group in seed_groups:

                    state_at_t = s_group[
                        s_group["elapsed_time"] <= t
                    ]

                    if not state_at_t.empty:

                        latest = state_at_t.iloc[-1]

                        best_feas = latest["best_feasible_obj"]
                        best_all = latest["best_obj"]

                        if pd.notna(best_feas):
                            vals_at_t.append(best_feas)
                            feas_flags.append(True)
                        elif pd.notna(best_all):
                            vals_at_t.append(best_all)
                            feas_flags.append(False)

                # Column name
                col_name = f"{variant}_{t}s"

                if not vals_at_t:
                    row_data[col_name] = "N/A"
                else:
                    mean_val = np.mean(vals_at_t)

                    # No feasible solution in any seed
                    if not any(feas_flags):
                        row_data[col_name] = f"{mean_val:.2f}*"
                    else:
                        row_data[col_name] = f"{mean_val:.2f}"

        rows.append(row_data)

    df_result = pd.DataFrame(rows)

    # Sort by regime (A/B) and instance size
    df_result["regime"] = (
        df_result["Instance"].str[-1].str.upper()
    )

    df_result["size_id"] = (
        df_result["Instance"]
        .str.extract(r"md(\d+)", expand=False)
        .astype(int)
    )

    df_result = (
        df_result
        .sort_values(["regime", "size_id"])
        .drop(columns=["regime", "size_id"])
        .reset_index(drop=True)
    )

    # Ensure column order:
    # Instance | Base_0.1s | Base_1s | ... | NR_0.1s | NR_1s | ...
    ordered_columns = (
        ["Instance"]
        + [f"Base_{t}s" for t in time_points]
        + [f"NR_{t}s" for t in time_points]
    )

    df_result = df_result[ordered_columns]

    return df_result


# ---------------------------------------------------------
# Generate and display table
# ---------------------------------------------------------

time_cutoffs = [0.1, 1, 2, 5, 10, 60]

df_all_instances_table = build_all_instances_time_table(
    df_conv_all,
    time_cutoffs
)

display(Markdown(
    "### Mean Best Objective over Time Across All Instances (20 Seeds)"
))

display(Markdown(
    "*Values with an asterisk (`*`) indicate that no feasible solution "
    "had been found yet by any seed at that time cutoff.*"
))

display(df_all_instances_table)

## 14. Convergence comparison by regime

In [ ]:
# ---------------------------------------------------------
# Convergence curves split by tight (A) / loose (B) time-window regime
# ---------------------------------------------------------

conv_for_curve["regime"] = conv_for_curve["instance"].str[-1].str.upper()

regime_curve = (
    conv_for_curve
    .dropna(subset=["feasible_RPD"])
    .groupby(["variant", "regime", "progress_bin"])
    ["feasible_RPD"]
    .median()
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

for ax, regime in zip(axes, ["A", "B"]):
    for variant in ["Base", "NR"]:
        g = regime_curve[(regime_curve["regime"] == regime) & (regime_curve["variant"] == variant)]
        ax.plot(g["progress_bin"], g["feasible_RPD"], label=variant, color=VARIANT_COLORS[variant])
    ax.set_title(f"Regime {regime}")
    ax.set_xlabel("Normalized search progress (%)")
    ax.legend()

axes[0].set_ylabel("Median best feasible RPD (%)")
fig.suptitle("ALNS Convergence by Regime: Base vs. NR")

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# Temporal Convergence Analysis (Baseline vs NR)
# ==============================================================================

time_intervals = [1.0, 2.0, 5.0, 15.0, 30.0, 60.0]

def get_temporal_convergence(df_convergence, intervals):
    results = []
    
    # Standardize variant names
    df_convergence = df_convergence.copy()
    df_convergence['variant'] = df_convergence['variant'].replace({'Base': 'Baseline'})
    
    grouped = df_convergence.groupby(['variant', 'instance', 'seed'])
    
    for (variant, instance, seed), group in grouped:
        group = group.sort_values('elapsed_time')
        
        for t in intervals:
            sub = group[group['elapsed_time'] <= t]
            
            if sub.empty:
                val_str = "N/A"
                numeric_val = np.nan
                is_feasible = False
            else:
                feasible_objs = sub['best_feasible_obj'].dropna()
                if not feasible_objs.empty:
                    numeric_val = feasible_objs.min()
                    val_str = f"{numeric_val:.3f}"
                    is_feasible = True
                else:
                    numeric_val = sub['best_obj'].min()
                    val_str = f"{numeric_val:.3f}*"
                    is_feasible = False
            
            results.append({
                'variant': variant,
                'instance': instance,
                'seed': seed,
                'time_limit_sec': t,
                'obj_value': numeric_val,
                'formatted_val': val_str,
                'is_feasible': is_feasible
            })
            
    return pd.DataFrame(results)

# Process convergence data
df_temp_conv = get_temporal_convergence(df_conv_all, time_intervals)

# ------------------------------------------------------------------------------
# 1. Summary Table Across All Runs (Mean Objective & Feasibility Rate)
# ------------------------------------------------------------------------------
summary_rows = []

for t in time_intervals:
    row = {'Time Interval (s)': f"{t}s"}
    
    for var in ['Baseline', 'NR']:
        sub_var = df_temp_conv[(df_temp_conv['variant'] == var) & (df_temp_conv['time_limit_sec'] == t)]
        
        if not sub_var.empty:
            mean_obj = sub_var['obj_value'].mean()
            feasible_pct = sub_var['is_feasible'].mean() * 100
            
            # Append asterisk if no run achieved feasibility in this interval
            asterisk = "" if sub_var['is_feasible'].any() else "*"
            row[f'{var} Mean Best Obj'] = f"{mean_obj:.3f}{asterisk}"
            row[f'{var} Feasible %'] = f"{feasible_pct:.1f}%"
        else:
            row[f'{var} Mean Best Obj'] = "N/A"
            row[f'{var} Feasible %'] = "0.0%"
            
    summary_rows.append(row)

df_summary_table = pd.DataFrame(summary_rows)

print("=== Temporal Convergence Summary Table ===")
display(df_summary_table)

# ------------------------------------------------------------------------------
# 2. Pivot Table per Instance (Averaged across seeds for each time limit)
# ------------------------------------------------------------------------------
def format_instance_cell(group):
    mean_val = group['obj_value'].mean()
    has_feasible = group['is_feasible'].any()
    asterisk = "" if has_feasible else "*"
    return f"{mean_val:.2f}{asterisk}" if pd.notna(mean_val) else "N/A"

instance_pivot = df_temp_conv.groupby(['instance', 'variant', 'time_limit_sec']).apply(
    format_instance_cell, include_groups=False
).unstack(level=['variant', 'time_limit_sec'])
instance_order = (
    df_runs_all[["instance", "regime", "size_id"]]
    .drop_duplicates()
    .sort_values(["regime", "size_id"])["instance"]
)
instance_pivot = instance_pivot.reindex(instance_order)

print("\n=== Detailed Instance-Level Temporal Convergence ===")
display(instance_pivot)

## 15. Time-to-best comparison

In [ ]:
# ---------------------------------------------------------
# Time (and % of run) to reach the best solution: Base vs. NR
# ---------------------------------------------------------

df_paired["best_time_pct_base"] = 100 * df_paired["best_time_sec_base"] / df_paired["elapsed_time_sec_base"]
df_paired["best_time_pct_nr"] = 100 * df_paired["best_time_sec_nr"] / df_paired["elapsed_time_sec_nr"]

time_to_best_comparison = (
    df_paired
    .groupby(["instance", "size_id", "regime"])
    .agg(
        base_mean_best_time_sec=("best_time_sec_base", "mean"),
        nr_mean_best_time_sec=("best_time_sec_nr", "mean"),
        base_mean_best_time_pct=("best_time_pct_base", "mean"),
        nr_mean_best_time_pct=("best_time_pct_nr", "mean"),
    )
    .reset_index()
    .sort_values("size_id")
)

display(time_to_best_comparison.round(3))

fig, ax = plt.subplots(figsize=(10, 6))
for variant, col in [("Base", "base_mean_best_time_sec"), ("NR", "nr_mean_best_time_sec")]:
    for regime, style in [("A", "-"), ("B", "--")]:
        g = time_to_best_comparison[time_to_best_comparison["regime"] == regime]
        ax.plot(g["size_id"], g[col], marker="o", linestyle=style,
                color=VARIANT_COLORS[variant], label=f"{variant} — Regime {regime}")

ax.set_xlabel("Instance size index")
ax.set_ylabel("Mean time to best solution (s)")
ax.set_title("Time to Reach the Best Solution: Base vs. NR")
ax.legend()

plt.tight_layout()
plt.show()


## 16. Export comparison tables

In [ ]:
# ---------------------------------------------------------
# Export the main comparison tables
# ---------------------------------------------------------

exports = {
    "paired_runs.csv": df_paired,
    "instance_comparison.csv": instance_comparison,
    "global_comparison.csv": global_comparison,
    "significance_tests.csv": significance_table,
    "win_loss_by_instance.csv": win_loss_by_instance.reset_index(),
    "time_to_best_comparison.csv": time_to_best_comparison,
}

for filename, table in exports.items():
    table.to_csv(COMPARISON_DIR / filename, index=False)

print(f"Exported {len(exports)} comparison tables to:")
print(COMPARISON_DIR.resolve())


# Summary and caveats

For a **Neighborhood Reduction ablation** subsection, the most informative artifacts are
likely:

- `instance_comparison.csv` / the objective slope chart — does NR change solution quality,
  and does the effect vary by instance size or time-window regime?
- `significance_tests.csv` — is any observed difference in objective or runtime
  statistically significant per instance?
- The win/loss breakdown — how consistent is the effect across seeds, not just on average?
- The aggregate convergence overlay — does NR converge faster/slower to a given RPD, even if
  the final RPD is similar?
- Runtime comparison — NR is expected to speed up each ALNS iteration (smaller neighborhoods
  to evaluate); check whether this translates into more iterations per unit time or a
  different runtime profile at 7200s.

**Caveats to double-check against your actual files:**

- The filename parser assumes optional underscores around the instance name (see the note in
  the introduction) to reconcile the slightly different naming used by the two SLURM scripts.
  If parsing fails or discovers 0 runs, that pattern needs revisiting for the actual filenames
  on disk.
- RPD is computed against a joint (Base ∪ NR) best-known reference per instance, not a
  per-variant reference, so RPD values here are **not directly comparable** to RPD values in
  the original Base-only notebook (which used a Base-only reference).
- The global pooled Wilcoxon test treats all 400 paired runs as exchangeable, which ignores
  the very different objective scales across instances; the per-instance tests are the more
  defensible statistical statement.